# 03 — Haiku reference, independent judging and consolidation

Run locally with the new Python 3.12 environment. No tunnel is required. Each request has an independent context. All returned judge text is retained; parsing only adds audit columns. API errors and unknown submissions are distinct from poor model answers. Haiku uses provider-default temperature (the API does not support temperature=0) and disabled thinking. The 128,000 reference output limit and 8,192 judge output limit are actual recorded wire parameters. Estimated USD cost uses the 2026-10-24 pricing snapshot; token usage and receipts are retained.

## 1. Local environment and watchdog

Locate project files and verify the independent audible monitor.

**Recovery / safe rerun:** On a kernel restart, rerun 1–2, then the interrupted stage. Use notebook 00 section 5 if the watchdog stopped.

**Cost:** No paid API requests.



In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

from streamlined.notebook import watchdog_ready
watchdog_ready(ROOT)

## 2. API connection

Read secret/Claude_API_Key.txt without displaying it. This metadata request verifies the exact model identifier without generating responses.

**Recovery / safe rerun:** Rerun 1–2 after an API-key change or kernel restart. The key is not written to notebook outputs.

**Cost:** No paid API requests.



In [ ]:
from streamlined.api import client, APIRun, haiku_jobs, judge_jobs
from streamlined.analysis import compile_tables, require_complete, consolidate_judgments, intervals
api = client(ROOT)
model_info = api.models.retrieve(POLICY['haiku_model'])
if model_info.id != POLICY['haiku_model']:
    raise ValueError('API returned a different model ID')
print('API access confirmed:',model_info.id)

## 3. Compile D and E before Haiku generation

D contains all expected raw/SFT rows, including pending/error/status rows. Three E files contain validation responses for technique, explanation and joint across all six models. All four adapters also receive the common rubric. Source keys and prompt hashes are checked.

**Recovery / safe rerun:** Rerun after any additional small-model results arrive. This updates views, never launches generation.

**Cost:** No paid API requests.



In [ ]:
compiled = compile_tables(ROOT)
print('D expected records:',len(compiled))
print('Tables:',ROOT/'results/compiled')

## 4. Haiku five-row pilot

Paid independent requests. Print each response, token usage and estimated cost. These are the first five missing IDs from the same 990-row ledger; the main run reuses them.

**Recovery / safe rerun:** Rerun 1–2 then reconstruct haiku here. Completed responses are skipped. An unknown synchronous request is held to avoid a duplicate charge; restore a saved receipt or investigate the API receipt before explicit retry.

**Cost:** No paid API requests.

Up to five paid Messages API requests; no quality-dependent retry.

In [ ]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
haiku.restore_receipts()
print(haiku.pilot(size=5))

## 5. Submit remaining Haiku references

Submit batches of 50. Persist intent before submission and remote batch ID immediately afterwards. Never rerun a completed prompt because of malformed or empty output.

**Recovery / safe rerun:** Rerun 1–2 and 5 after restart. Unknown submissions remain held; reconcile after remote completion. Use section 10 for requests with confirmed API failures.

**Cost:** Paid Batch API requests for missing rows.



In [ ]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
print(haiku.reconcile())
print(haiku.submit(batch_size=50))

## 6. Poll and save Haiku responses

Polling saves each returned response locally. A poll window ending does not mean all batches have finished. No new paid work is submitted here.

**Recovery / safe rerun:** Rerun 1–2 then 6 until completed=990, or use 10 for confirmed failures. Rerun 5 to reconcile an unknown batch.

**Cost:** No paid API requests.



In [ ]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
print(haiku.poll(max_seconds=3600))

## 7. Join Haiku references and freeze judge jobs

Recompile D/E with corresponding Haiku responses. Require all expected candidate rows to be completed or explicitly context-unsupported before freezing requests. A single candidate is judged in each context; raw and SFT never share a judging request. Haiku self-assessment runs separately, once per validation/test prompt.

**Recovery / safe rerun:** After kernel restart rerun 1–2 and 7. If candidate outputs are missing, finish those model notebooks first. Never edit prompt/rubric settings inside an existing paid ledger.

**Cost:** No paid API requests.



In [ ]:
compiled = compile_tables(ROOT)
require_complete(compiled)
runs = {'common':APIRun(ROOT,'common',judge_jobs(compiled),api)}
for condition in ('technique','explanation','joint'):
    subset = [r for r in compiled if r['condition']==condition and r['split']=='validation']
    runs[condition] = APIRun(ROOT,condition,judge_jobs(subset,condition),api)
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
self_rows = [r for r in haiku.store.rows.values() if r.get('split') in ('validation','test')]
runs['haiku_self'] = APIRun(ROOT,'haiku_self',judge_jobs(self_rows,self_assessment=True),api)
print({name:run.counts() for name,run in runs.items()})

## 8. Submit common, conditional and self-assessment judgments

All six rubric dimensions are retained. Specific rubric scores are separate constructs and must not be averaged with unlike common metrics.

**Recovery / safe rerun:** Rerun 1–2, 7 and 8 after interruption. Completed/submitted jobs are skipped; unknown submissions stay held.

**Cost:** No paid API requests.

Paid batch judging. Counts are printed before each group is submitted.

In [ ]:
for name, run in runs.items():
    print(name,run.reconcile())
    print(run.submit(batch_size=50))

## 9. Poll judgments and consolidate

Malformed judge outputs remain completed observations. Missing extracted scores stay missing, never zero. Consolidation retains raw judge text, candidate text, usage and audit data.

**Recovery / safe rerun:** Rerun 1–2, 7 and 9 until no submitted jobs remain. Section 10 is only for confirmed API failures.

**Cost:** No paid API requests.



In [ ]:
for name,run in runs.items():
    print(name,run.poll(max_seconds=3600))
judged = consolidate_judgments(ROOT)
print('Consolidated rows:',len(judged))

## 10. Resume missing or confirmed failed requests

This is an explicit retry of process/API failures. It never rejudges a received response for formatting or quality. Unknown synchronous requests require an audited resolution.

**Recovery / safe rerun:** After restart rerun 1–2, rebuild haiku (5/6) or runs (7), then 10. Repeat polling, not submission, for pending work.

**Cost:** No paid API requests.

Paid retries only for pending/rejected/confirmed failed requests.

In [ ]:
RETRY_TARGET = 'haiku'  # or common, technique, explanation, joint, haiku_self
target = haiku if RETRY_TARGET=='haiku' else runs[RETRY_TARGET]
print(target.reconcile())
print(target.submit(batch_size=50,retry_failed=True))
print(target.poll(max_seconds=3600))

## 10a. Optional resolution of an unknown synchronous pilot

Use only after checking the saved receipt/API outcome. A lost synchronous response is not always recoverable. Retrying it may create a duplicate charge. Batch uncertainty must use reconciliation instead; this cell refuses batch records.

**Recovery / safe rerun:** Safe default does nothing. Reconstruct the target run first. A documented resolution enables section 10 retry or restores a receipt without another API call.

**Cost:** No paid API requests.



In [ ]:
RESOLVE_UNKNOWN = False
RECORD_ID = ''
RESOLUTION_REASON = ''
RECOVERED_RECEIPT = None  # Path to a recovered complete API message JSON, if available
ALLOW_POSSIBLE_DUPLICATE_CHARGE = False
if RESOLVE_UNKNOWN:
    target.resolve_unknown_pilot(RECORD_ID,RESOLUTION_REASON,
        receipt_path=RECOVERED_RECEIPT,allow_retry=ALLOW_POSSIBLE_DUPLICATE_CHARGE)

## 11. Final CSVs and grouped paired intervals

The consolidated CSV is the new charts input. Matched comparisons require identical prompt hashes. Bootstrap scenario-step groups; report available pairs per metric and keep nulls. Comparisons are exploratory, without multiplicity-adjusted claims.

**Recovery / safe rerun:** Safe to rerun after any recovered results. It does not submit requests or change source responses.

**Cost:** No paid API requests.



In [ ]:
compile_tables(ROOT)
print('Judgments:',len(consolidate_judgments(ROOT)))
summary = intervals(ROOT)
print('Comparison rows:',len(summary))
print('Charts input:',ROOT/'results/compiled/all_judgments.csv')